# Week 07 - Image Segmentation

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Segment images by **thresholding**: global, Otsu and adaptive.
- Detect boundaries with **edge-based segmentation** (Canny) and Hough transforms.
- Segment by **regions** using connected components, flood fill and **watershed**.
- Evaluate segmentation quality and choose the right method for a task.

### What segmentation means
Segmentation partitions an image into meaningful regions: foreground vs background, or individual objects. It is the bridge between low-level pixels and high-level recognition (Weeks 8-11).

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python numpy matplotlib scipy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap=cmap or (None if img.ndim == 3 else "gray"))
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

img = cv2.imread("resources/images/money_counter.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Shape:", img.shape)

## 2. Guided example - thresholding
- **Global**: one fixed threshold for the whole image.
- **Otsu**: automatically picks the threshold that best separates two intensity classes.
- **Adaptive**: a different threshold per neighbourhood, for uneven lighting.

In [ ]:
_, th_global = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
otsu_val, th_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
th_adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY, 31, 5)
print("Otsu threshold value:", otsu_val)
show(th_global, th_otsu, th_adaptive,
     titles=["Global (127)", f"Otsu ({int(otsu_val)})", "Adaptive mean"])

## 3. Guided example - edge-based segmentation with Canny
Canny has four stages: gradient computation, non-maximum suppression, double thresholding and hysteresis. The two thresholds control which edges survive.

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 1.4)
edges_low = cv2.Canny(blur, 30, 90)
edges_mid = cv2.Canny(blur, 80, 170)
edges_high = cv2.Canny(blur, 150, 250)
show(edges_low, edges_mid, edges_high, titles=["30/90", "80/170", "150/250"])

## 4. Guided example - Hough transform for lines and circles
Hough transforms accumulate votes for parametric shapes. This is powerful for detecting manufactured parts (straight edges, circular holes).

In [ ]:
# Synthetic shape image
canvas = np.zeros((400, 400), np.uint8)
cv2.line(canvas, (40, 40), (360, 120), 255, 3)
cv2.line(canvas, (40, 360), (360, 200), 255, 3)
cv2.circle(canvas, (200, 280), 60, 255, 3)
cv2.circle(canvas, (120, 200), 30, 255, 3)

edges = cv2.Canny(canvas, 50, 150)
lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=80,
                        minLineLength=60, maxLineGap=10)
circles = cv2.HoughCircles(canvas, cv2.HOUGH_GRADIENT, dp=1, minDist=40,
                           param1=100, param2=30, minRadius=20, maxRadius=80)

vis = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
if lines is not None:
    for l in lines:
        x1, y1, x2, y2 = l[0]
        cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
if circles is not None:
    for c in np.round(circles[0]).astype(int):
        cv2.circle(vis, (c[0], c[1]), c[2], (0, 255, 0), 2)
print("Lines detected:", 0 if lines is None else len(lines))
print("Circles detected:", 0 if circles is None else len(circles[0]))
show(canvas, vis, titles=["Synthetic shapes", "Hough lines (red) & circles (green)"])

## 5. Guided example - region-based segmentation
`connectedComponentsWithStats` labels connected regions and gives area/bounding boxes. `floodFill` grows a region from a seed. **Watershed** separates touching objects using markers.

In [ ]:
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))

n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
print("Connected components (incl. background):", n)

vis = img.copy()
count = 0
for i in range(1, n):
    area = stats[i, cv2.CC_STAT_AREA]
    if area < 200:      # ignore noise
        continue
    count += 1
    x, y, w, h = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 0, 255), 2)
    cv2.putText(vis, str(count), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
print("Significant objects:", count)
show(vis, titles=["Region segmentation"])

In [ ]:
# Watershed to split touching objects
_, sure_fg = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
sure_fg = cv2.morphologyEx(sure_fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
sure_fg = cv2.dilate(sure_fg, np.ones((3, 3), np.uint8), iterations=1)
sure_bg = cv2.dilate(sure_fg, np.ones((3, 3), np.uint8), iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)

_, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0
markers = cv2.watershed(img, markers)

vis_w = img.copy()
vis_w[markers == -1] = [0, 0, 255]
print("Watershed regions (max label):", markers.max())
show(vis_w, titles=["Watershed boundaries"])

## 6. Exercise (complete the code)

Write a segmentation function `segment(img)` that:
1. Converts to grayscale and applies a Gaussian blur.
2. Applies **Otsu** thresholding.
3. Removes noise with morphological opening.
4. Returns a clean binary mask.
Then apply it to `resources/images/money_counter.png` and report the number of objects with area > 500.

In [ ]:
def segment(image):
    # TODO: implement the pipeline
    return None

# mask = segment(img)
# show(mask) and count objects

## 7. Challenge (independent)

Compare **threshold-based** and **edge-based** segmentation on the same image. Build a small table reporting, for each method:
- number of regions found,
- whether touching objects were split,
- sensitivity to lighting.

Then state which method you would use for a coin-counting machine and why.

In [ ]:
# Your code here


## 8. Reflection
1. When would adaptive thresholding clearly beat global thresholding?
2. Why does Canny need two thresholds instead of one?
3. Watershed often over-segments. What can you do to reduce over-segmentation?

## 9. Visual summary

In [ ]:
import sys
sys.path.append("resources/scripts")
from cvhelpers import concept_map

concept_map([
    "Segmentation goal: partition into meaningful regions",
    "Thresholding: global, Otsu, adaptive",
    "Edge-based: Canny + Hough for lines/circles",
    "Region-based: connected components, flood fill, watershed",
    "Evaluation: does the split match the task?",
    "Deep learning (Week 11) replaces hand-crafted rules"
], title="Segmentation toolbox")

## 10. Interactive exploration - Canny thresholds

The two Canny thresholds control which edges survive. A low `t1` admits weak edges (and noise); a high `t2` keeps only strong edges.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def canny_demo(t1=50, t2=150):
    e = cv2.Canny(blur, t1, t2)
    show(e, titles=[f"Canny {t1}/{t2}"])

interact(canny_demo,
         t1=widgets.IntSlider(min=10, max=200, step=10, value=50),
         t2=widgets.IntSlider(min=50, max=300, step=10, value=150))

## 11. Check your understanding (Q&A)

<details><summary><b>Q1. When does adaptive thresholding clearly beat global thresholding?</b></summary>

When illumination is uneven (shadows, vignetting), because it computes a local threshold per neighbourhood instead of one value for the whole image.

</details>

<details><summary><b>Q2. Why does Canny use two thresholds?</b></summary>

To apply hysteresis: weak edges (above t1) are kept only if they connect to strong edges (above t2), which reduces spurious edges.

</details>

<details><summary><b>Q3. Watershed often over-segments - how can this be reduced?</b></summary>

Use better markers (e.g. distance transform + threshold, or h-maxima) so each object starts with a single seed.

</details>

## 12. Further reading & self-exploration
- OpenCV thresholding: https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html
- OpenCV Canny: https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html
- OpenCV Hough lines and circles: https://docs.opencv.org/4.x/d6/d10/tutorial_py_houghlines.html
- OpenCV watershed: https://docs.opencv.org/4.x/d3/db4/tutorial_py_watershed.html
- scikit-image segmentation: https://scikit-image.org/docs/stable/user_guide/tutorial_segmentation.html
- Wikipedia - Watershed transform: https://en.wikipedia.org/wiki/Watershed_(image_processing)

**Try next:** build a coin counter that combines Otsu + morphology + watershed and reports the count.

## 13. Key takeaways
- Thresholding is the simplest segmentation: global, Otsu, adaptive.
- Canny + Hough extracts edges and parametric shapes.
- Region methods handle touching objects.
- The "right" segmentation depends on the application.
